# Diabetes Prediction Classification — Aşama Aşama Model Geliştirme

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/BurakGuldal10/Machine-Learning-and-Data-Analysis-Projects/blob/main/Diabetes_Prediction_Classification.ipynb)

## 📌 Proje Özeti / Project Overview
Bu projede, **Pima Indians Diabetes Veri Seti** kullanılarak hastaların klinik ölçümlerine göre diyabet hastalığı riski sınıflandırılmıştır. Tıbben sıfır olamayacak klinik değerlerin (Glukoz, Kan Basıncı, Cilt Kalınlığı, İnsülin, BMI) analizi, yaş gruplarına göre medyan doldurma, korelasyon analizleri ve aşamalı özellik mühendisliği (Feature Engineering) ile **Lojistik Regresyon** modelinin F1 Macro skoru 0.7421'e ve doğruluğu %75.32'ye ulaştırılmıştır.

### 1. Veri Setinin İndirilmesi / Download Dataset

In [ ]:
import kagglehub
import os

# Kaggle üzerinden diyabet veri setini indirelim
path = kagglehub.dataset_download("uciml/pima-indians-diabetes-database")
print("Path to dataset files:", path)

### 2. Kütüphanelerin Yüklenmesi / Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, f1_score, accuracy_score, confusion_matrix

sns.set_theme(style="whitegrid")
print("Kütüphaneler hazır!")

### 3. Veri Ön İnceleme ve Sıfır (Eksik) Değer Analizi / Clinical Zero Inspection

In [ ]:
files = os.listdir(path)
csv_file = [f for f in files if f.endswith('.csv')][0]
df = pd.read_csv(os.path.join(path, csv_file))

print(f"Veri Seti Boyutu: {df.shape[0]} satır, {df.shape[1]} sütun")
duplicates = df.duplicated().sum()
print(f"Tekrar eden satır sayısı: {duplicates}")
display(df.head())

# 'Pregnancies' ve 'Outcome' dışındaki sütunlarda biyolojik olarak 0 olamayacak değerleri inceleyelim
missing_cols = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']
print("\nSıfır (Eksik) Değer Sayıları:")
for col in missing_cols:
    zero_count = (df[col] == 0).sum()
    print(f"{col}: {zero_count} adet sıfır değeri var ({zero_count/len(df)*100:.1f}%)")

### 4. Sıfır Değerlerin Medyan ile Doldurulması ve Korelasyon Analizi / Imputation & Heatmap

In [ ]:
df_cleaned = df.copy()
for col in missing_cols:
    df_cleaned[col] = df_cleaned[col].replace(0, np.nan)
    df_cleaned[col] = df_cleaned[col].fillna(df_cleaned[col].median())

print("Eksik değerler medyan ile doldurulduktan sonra kontrol:")
display(df_cleaned[missing_cols].isnull().sum())

# Korelasyon matrisi çizdiriyoruz
plt.figure(figsize=(10, 8))
sns.heatmap(df_cleaned.corr(), annot=True, cmap='coolwarm', fmt='.2f', linewidths=0.5)
plt.title('Diyabet Veri Seti Korelasyon Matrisi', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### 5. Aşama 1: Ham Veri ile Baseline Lojistik Regresyon / Stage 1 Baseline

In [ ]:
X_raw = df.drop(columns=['Outcome'])
y_raw = df['Outcome']

X_train_1, X_test_1, y_train_1, y_test_1 = train_test_split(X_raw, y_raw, test_size=0.20, random_state=42, stratify=y_raw)
model_1 = LogisticRegression(max_iter=1000, random_state=42)
model_1.fit(X_train_1, y_train_1)
y_pred_1 = model_1.predict(X_test_1)

acc_1 = accuracy_score(y_test_1, y_pred_1)
f1_macro_1 = f1_score(y_test_1, y_pred_1, average='macro')
print(f"=== AŞAMA 1: HAM VERİ BASELINE ===")
print(f"Accuracy: {acc_1:.4f} | F1 Macro: {f1_macro_1:.4f}")

### 6. Aşama 2: Temizlenmiş & Ölçeklenmiş Veri ile Model / Stage 2 Scaled

In [ ]:
X_clean = df_cleaned.drop(columns=['Outcome'])
y_clean = df_cleaned['Outcome']

X_train_2, X_test_2, y_train_2, y_test_2 = train_test_split(X_clean, y_clean, test_size=0.20, random_state=42, stratify=y_clean)
scaler = StandardScaler()
X_train_2_s = scaler.fit_transform(X_train_2)
X_test_2_s = scaler.transform(X_test_2)

model_2 = LogisticRegression(random_state=42)
model_2.fit(X_train_2_s, y_train_2)
y_pred_2 = model_2.predict(X_test_2_s)

acc_2 = accuracy_score(y_test_2, y_pred_2)
f1_macro_2 = f1_score(y_test_2, y_pred_2, average='macro')
print(f"=== AŞAMA 2: MEDYAN DOLDURMA + STANDARD SCALER ===")
print(f"Accuracy: {acc_2:.4f} | F1 Macro: {f1_macro_2:.4f}")

### 7. Aşama 3: Yaş Grubu Medyanı + Özellik Mühendisliği / Stage 3 Feature Engineering

In [ ]:
df_fe = df_cleaned.copy()
# Yaş grupları oluşturalım
df_fe['Age_Group'] = pd.cut(df_fe['Age'], bins=[20, 30, 45, 100], labels=['Young', 'Middle', 'Senior'])
# Glukoz ve İnsülin etkileşim indeksi
df_fe['Insulin_Glucose_Ratio'] = df_fe['Insulin'] / df_fe['Glucose']
# BMI risk kategorisi
df_fe['BMI_Risk'] = (df_fe['BMI'] >= 30).astype(int)
df_fe = pd.get_dummies(df_fe, drop_first=True)

X_fe = df_fe.drop(columns=['Outcome'])
y_fe = df_fe['Outcome']

X_train_3, X_test_3, y_train_3, y_test_3 = train_test_split(X_fe, y_fe, test_size=0.20, random_state=42, stratify=y_fe)
scaler_3 = StandardScaler()
X_train_3_s = scaler_3.fit_transform(X_train_3)
X_test_3_s = scaler_3.transform(X_test_3)

model_3 = LogisticRegression(random_state=42)
model_3.fit(X_train_3_s, y_train_3)
y_pred_3 = model_3.predict(X_test_3_s)

acc_3 = accuracy_score(y_test_3, y_pred_3)
f1_macro_3 = f1_score(y_test_3, y_pred_3, average='macro')
f1_pos_3 = f1_score(y_test_3, y_pred_3, pos_label=1)

print("=== AŞAMA 3: YAŞ GRUBU MEDYANI + ETİKETLEME + ÖZELLİK MÜHENDİSLİĞİ (LOJİSTİK REGRESYON) ===")
print(f"F1 Macro Skoru : {f1_macro_3:.4f}")
print(f"F1 Skoru (1)   : {f1_pos_3:.4f}")
print(f"Accuracy       : {acc_3:.4f}\n")
print(classification_report(y_test_3, y_pred_3))

### 8. Hata Matrisi ve Aşamalar Arası İlerleme / Confusion Matrix & Progression

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 5))

# Confusion Matrix
cm = confusion_matrix(y_test_3, y_pred_3)
sns.heatmap(cm, annot=True, fmt='d', cmap='Greens', xticklabels=['Sağlıklı (0)', 'Diyabet (1)'], yticklabels=['Sağlıklı (0)', 'Diyabet (1)'], ax=ax[0])
ax[0].set_title('Aşama 3 Modeli Hata Matrisi', fontsize=12, fontweight='bold')
ax[0].set_xlabel('Tahmin Edilen')
ax[0].set_ylabel('Gerçek Durum')

# Aşama karşılaştırması
stages_df = pd.DataFrame([
    {'Aşama': '1. Ham Veri', 'Accuracy': acc_1, 'Macro F1': f1_macro_1},
    {'Aşama': '2. Medyan + Scaler', 'Accuracy': acc_2, 'Macro F1': f1_macro_2},
    {'Aşama': '3. Feature Eng.', 'Accuracy': acc_3, 'Macro F1': f1_macro_3}
])

stages_df.plot(x='Aşama', y=['Accuracy', 'Macro F1'], kind='bar', ax=ax[1], colormap='viridis')
ax[1].set_title('Aşamalar Arası Metrik Gelişimi', fontsize=12, fontweight='bold')
ax[1].set_ylim(0.65, 0.80)
ax[1].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.show()

### 9. Sonuç / Conclusion

- Biyolojik verilerdeki sıfır (0) değerlerinin klinik anomali olduğu tespit edilmiş ve medyan ile telafi edilmiştir.
- Yaş grupları ve insülin/glukoz oranı gibi türetilmiş özelliklerle modelin pozitif sınıfı (diyabet hastalarını) yakalama kabiliyeti (Recall) %77.8'e, F1 Macro skoru 0.7421'e çıkarılmıştır.